In [ ]:
#@title Cell 02.1 - Notebook overview
# This cell defines the purpose and fixed analysis plan for Notebook 02.

from IPython.display import display, Markdown

display(Markdown(r"""
# Notebook 02: blaTEM-1 upper-MIC nearest-neighbour assessment

## Purpose

Notebook 02 assesses whether each of the 16 upper-MIC blaTEM-1-only pathogens
has chromosomally close lower-MIC comparators among the remaining 160 pathogens.

For each upper-MIC pathogen, all 160 remaining pathogens are ranked by the same
K-derived chromosomal distance used in Notebook 01. The 10 nearest neighbours
are then examined together with their MIC values and log2(MIC) differences.

No final matched controls are selected in this notebook.

## Notebook structure

Notebook 02 contains 7 code cells:

1. notebook overview;
2. locate the public repository and locate fixed inputs;
3. load and validate the 176-pathogen cohort, K and 16/160 groups;
4. calculate distances from each upper-MIC pathogen to all 160 comparators;
5. extract the 10 nearest neighbours for each upper-MIC pathogen;
6. summarize neighbour quality and MIC separation;
7. save final QC, provenance and outputs.
"""))

print(
    "Notebook 02 overview complete.\n"
    "Transition: Cell 02.2 will locate the public repository and locate the fixed "
    "Notebook 01 and previous-project inputs."
)


In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 02.2 - Mount Drive and locate fixed inputs
# This cell defines the previous-project read-only inputs, Notebook 01 outputs,
# and Notebook 02 output directories.

from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
PREVIOUS_PROJECT_ROOT = _previous_project_root(PROJECT_ROOT)
PROJECT_ROOT = _repo_root()

# Previous-project fixed inputs.
NOTEBOOK03_DIRECTORY = PREVIOUS_PROJECT_ROOT / '02_Data_Preparation' / 'Notebook03'
NOTEBOOK04_DIRECTORY = PREVIOUS_PROJECT_ROOT / '04_Population_Structure' / 'Notebook04'

PATHOGEN_INDEX_PATH = (
    NOTEBOOK03_DIRECTORY
    / '03_ceftazidime_pathogen_index.csv'
)
RELATEDNESS_MATRIX_PATH = (
    NOTEBOOK04_DIRECTORY
    / '04_genome_wide_relatedness_matrix.npz'
)
NOTEBOOK04_QC_PATH = (
    NOTEBOOK04_DIRECTORY
    / '04_population_structure_qc_summary.csv'
)

# Notebook 01 outputs used as fixed empirical groups.
HIGH_MIC_PATH = (
    PROJECT_ROOT
    / '02_Data_New'
    / 'High_MIC_16'
    / '01_high_MIC_16_pathogens.csv'
)
COMPARISON_PATH = (
    PROJECT_ROOT
    / '02_Data_New'
    / 'Comparison_160'
    / '01_comparison_160_pathogens.csv'
)
TEM176_PATH = (
    PROJECT_ROOT
    / '05_Results'
    / 'Tables'
    / '01_blaTEM-1_only_176_pathogens.csv'
)

# Notebook 02 outputs.
RELATEDNESS_DIRECTORY = PROJECT_ROOT / '04_Intermediate' / 'Relatedness'
TABLE_DIRECTORY = PROJECT_ROOT / '05_Results' / 'Tables'
FIGURE_DIRECTORY = PROJECT_ROOT / '05_Results' / 'Figures'
STATISTICAL_OUTPUT_DIRECTORY = PROJECT_ROOT / '05_Results' / 'Statistical_Outputs'

for directory in [
    RELATEDNESS_DIRECTORY,
    TABLE_DIRECTORY,
    FIGURE_DIRECTORY,
    STATISTICAL_OUTPUT_DIRECTORY,
]:
    directory.mkdir(parents=True, exist_ok=True)

required_inputs = [
    PATHOGEN_INDEX_PATH,
    RELATEDNESS_MATRIX_PATH,
    NOTEBOOK04_QC_PATH,
    HIGH_MIC_PATH,
    COMPARISON_PATH,
    TEM176_PATH,
]

missing_inputs = [str(path) for path in required_inputs if not path.exists()]

if missing_inputs:
    raise FileNotFoundError(
        'Required input(s) were not found:\n' + '\n'.join(missing_inputs)
    )

print(f'Previous project: {PREVIOUS_PROJECT_ROOT}')
print(f'Current project:  {PROJECT_ROOT}')
print('All required fixed inputs were found.')
print('Previous-project files will be treated as read only.')
print(
    'Transition: Cell 02.3 will load and validate the 176-pathogen cohort, '
    'the 16/160 groups and the aligned K matrix.'
)


In [ ]:
#@title Cell 02.3 - Load and validate cohort, groups and K
# This cell loads the Notebook 01 empirical groups and reconstructs the aligned
# 176 x 176 K subset using the original 1,672-pathogen K ordering.

EXPECTED_PATHOGENS = 1672
EXPECTED_RELATEDNESS_SNPS = 27988
EXPECTED_TEM_ONLY = 176
EXPECTED_HIGH_MIC = 16
EXPECTED_COMPARISON = 160

pathogen_index = pd.read_csv(
    PATHOGEN_INDEX_PATH,
    dtype={
        'biosample': str,
        'assembly_accession': str,
    },
)

high16 = pd.read_csv(
    HIGH_MIC_PATH,
    dtype={
        'biosample': str,
        'assembly_accession': str,
    },
)

comparison160 = pd.read_csv(
    COMPARISON_PATH,
    dtype={
        'biosample': str,
        'assembly_accession': str,
    },
)

tem176 = pd.read_csv(
    TEM176_PATH,
    dtype={
        'biosample': str,
        'assembly_accession': str,
    },
)

relatedness_archive = np.load(RELATEDNESS_MATRIX_PATH)
K = relatedness_archive['relatedness_matrix'].astype(np.float64)
relatedness_rows = relatedness_archive['notebook03_row_index'].astype(np.int64)

notebook04_qc = pd.read_csv(NOTEBOOK04_QC_PATH)

if len(pathogen_index) != EXPECTED_PATHOGENS:
    raise ValueError(
        f'Expected {EXPECTED_PATHOGENS} pathogen rows; found {len(pathogen_index)}.'
    )

if K.shape != (EXPECTED_PATHOGENS, EXPECTED_PATHOGENS):
    raise ValueError(
        f'Expected K shape {(EXPECTED_PATHOGENS, EXPECTED_PATHOGENS)}; found {K.shape}.'
    )

if not np.array_equal(
    relatedness_rows,
    np.arange(EXPECTED_PATHOGENS, dtype=np.int64),
):
    raise ValueError('K row order does not match Notebook 03 row identifiers.')

if len(tem176) != EXPECTED_TEM_ONLY:
    raise ValueError(
        f'Expected {EXPECTED_TEM_ONLY} blaTEM-1-only pathogens; found {len(tem176)}.'
    )

if len(high16) != EXPECTED_HIGH_MIC:
    raise ValueError(
        f'Expected {EXPECTED_HIGH_MIC} upper-MIC pathogens; found {len(high16)}.'
    )

if len(comparison160) != EXPECTED_COMPARISON:
    raise ValueError(
        f'Expected {EXPECTED_COMPARISON} comparison pathogens; found {len(comparison160)}.'
    )

for name, frame in [
    ('pathogen_index', pathogen_index),
    ('tem176', tem176),
    ('high16', high16),
    ('comparison160', comparison160),
]:
    if 'biosample' not in frame.columns:
        raise ValueError(f'{name} does not contain biosample.')
    if frame['biosample'].duplicated().any():
        raise ValueError(f'{name} contains duplicate BioSamples.')

high_ids = set(high16['biosample'])
comparison_ids = set(comparison160['biosample'])
tem_ids = tem176['biosample'].tolist()

if high_ids & comparison_ids:
    raise ValueError('The 16 and 160 empirical groups overlap.')

if high_ids | comparison_ids != set(tem_ids):
    raise ValueError(
        'The 16 and 160 groups do not exactly partition the 176-pathogen cohort.'
    )

pathogen_index = (
    pathogen_index
    .sort_values('notebook03_row_index')
    .reset_index(drop=True)
)

if not np.array_equal(
    pathogen_index['notebook03_row_index'].to_numpy(dtype=np.int64),
    np.arange(EXPECTED_PATHOGENS, dtype=np.int64),
):
    raise ValueError('Notebook 03 row indices are not the expected 0..1671 order.')

full_id_to_index = {
    biosample: idx
    for idx, biosample in enumerate(pathogen_index['biosample'])
}

missing_from_k = [
    biosample
    for biosample in tem_ids
    if biosample not in full_id_to_index
]

if missing_from_k:
    raise ValueError(
        'At least one blaTEM-1-only BioSample is absent from the K index.'
    )

tem_full_indices = np.array(
    [full_id_to_index[biosample] for biosample in tem_ids],
    dtype=np.int64,
)

K_tem = K[np.ix_(tem_full_indices, tem_full_indices)]

if not np.isfinite(K_tem).all():
    raise ValueError('The 176-pathogen K subset contains a non-finite value.')

if not np.allclose(K_tem, K_tem.T, rtol=1e-10, atol=1e-10):
    raise ValueError('The 176-pathogen K subset is not symmetric.')

qc_lookup = dict(
    zip(
        notebook04_qc['metric'].astype(str),
        notebook04_qc['value'],
    )
)

retained_snps = int(float(qc_lookup['Complete biallelic core SNPs retained']))

if retained_snps != EXPECTED_RELATEDNESS_SNPS:
    raise ValueError(
        f'Expected {EXPECTED_RELATEDNESS_SNPS} SNPs used for K; found {retained_snps}.'
    )

tem_id_to_local = {
    biosample: idx
    for idx, biosample in enumerate(tem_ids)
}

high_local_indices = np.array(
    [tem_id_to_local[biosample] for biosample in high16['biosample']],
    dtype=np.int64,
)
comparison_local_indices = np.array(
    [tem_id_to_local[biosample] for biosample in comparison160['biosample']],
    dtype=np.int64,
)

input_summary = pd.DataFrame([
    {'component': 'Previous-project pathogens', 'value': len(pathogen_index)},
    {'component': 'SNPs used for K', 'value': retained_snps},
    {'component': 'blaTEM-1-only pathogens', 'value': len(tem176)},
    {'component': 'Upper-MIC pathogens', 'value': len(high16)},
    {'component': 'Comparison pathogens', 'value': len(comparison160)},
])

display(input_summary)

print('Input alignment and QC passed.')
print(
    'Transition: Cell 02.4 will calculate K-derived chromosomal distances '
    'from each upper-MIC pathogen to all 160 comparison pathogens.'
)


In [ ]:
#@title Cell 02.4 - Calculate upper-MIC to comparison chromosomal distances
# This cell calculates the same K-derived distance used in Notebook 01 from
# each of the 16 upper-MIC pathogens to all 160 comparison pathogens.

K_diag = np.diag(K_tem)

distance_squared = (
    K_diag[:, None]
    + K_diag[None, :]
    - 2.0 * K_tem
)

distance_squared = np.clip(
    distance_squared,
    0.0,
    None,
)

distance_matrix = np.sqrt(distance_squared)
np.fill_diagonal(distance_matrix, 0.0)

if not np.allclose(
    distance_matrix,
    distance_matrix.T,
    rtol=1e-10,
    atol=1e-10,
):
    raise ValueError('K-derived distance matrix is not symmetric.')

if not np.isfinite(distance_matrix).all():
    raise ValueError('K-derived distance matrix contains a non-finite value.')

high_lookup = high16.set_index('biosample')
comparison_lookup = comparison160.set_index('biosample')

distance_rows = []

for upper_biosample in high16['biosample']:
    i = tem_id_to_local[upper_biosample]
    upper_row = high_lookup.loc[upper_biosample]
    upper_log2_mic = float(upper_row['log2_mic'])

    for comparison_biosample in comparison160['biosample']:
        j = tem_id_to_local[comparison_biosample]
        comparison_row = comparison_lookup.loc[comparison_biosample]
        comparison_log2_mic = float(comparison_row['log2_mic'])

        distance_rows.append({
            'upper_biosample': upper_biosample,
            'upper_assembly_accession': upper_row['assembly_accession'],
            'upper_log2_mic': upper_log2_mic,
            'upper_mic_mg_L': float(upper_row['observed_mic']),
            'comparison_biosample': comparison_biosample,
            'comparison_assembly_accession': comparison_row['assembly_accession'],
            'comparison_log2_mic': comparison_log2_mic,
            'comparison_mic_mg_L': float(comparison_row['observed_mic']),
            'delta_log2_mic_upper_minus_comparison': (
                upper_log2_mic - comparison_log2_mic
            ),
            'K_relatedness': float(K_tem[i, j]),
            'K_derived_distance': float(distance_matrix[i, j]),
        })

all_distances = pd.DataFrame(distance_rows)

expected_pairs = EXPECTED_HIGH_MIC * EXPECTED_COMPARISON

if len(all_distances) != expected_pairs:
    raise ValueError(
        f'Expected {expected_pairs} upper/comparison pairs; found {len(all_distances)}.'
    )

ALL_DISTANCE_PATH = (
    RELATEDNESS_DIRECTORY
    / '02_high_MIC_16_to_comparison_160_all_distances.csv.gz'
)

all_distances.to_csv(
    ALL_DISTANCE_PATH,
    index=False,
    compression='gzip',
)

distance_summary = pd.DataFrame([
    {'metric': 'Upper-MIC pathogens', 'value': EXPECTED_HIGH_MIC},
    {'metric': 'Comparison pathogens', 'value': EXPECTED_COMPARISON},
    {'metric': 'Upper/comparison pairs', 'value': len(all_distances)},
    {'metric': 'Minimum K-derived distance', 'value': all_distances['K_derived_distance'].min()},
    {'metric': 'Median K-derived distance', 'value': all_distances['K_derived_distance'].median()},
    {'metric': 'Maximum K-derived distance', 'value': all_distances['K_derived_distance'].max()},
])

display(distance_summary)

print(f'Saved: {ALL_DISTANCE_PATH}')
print(
    'Transition: Cell 02.5 will rank the 160 comparison pathogens for each '
    'upper-MIC pathogen and extract the 10 nearest chromosomal neighbours.'
)


In [ ]:
#@title Cell 02.5 - Extract the 10 nearest neighbours for each upper-MIC pathogen
# This cell ranks all 160 comparison pathogens by K-derived distance for each
# upper-MIC pathogen and retains the 10 nearest neighbours without imposing
# an MIC-difference threshold.

NEIGHBOURS_PER_UPPER = 10

nearest10 = (
    all_distances
    .sort_values(
        [
            'upper_biosample',
            'K_derived_distance',
            'comparison_biosample',
        ],
        ascending=[True, True, True],
    )
    .groupby(
        'upper_biosample',
        sort=False,
        group_keys=False,
    )
    .head(NEIGHBOURS_PER_UPPER)
    .copy()
)

nearest10['neighbour_rank'] = (
    nearest10
    .groupby('upper_biosample')
    .cumcount()
    + 1
)

nearest10 = nearest10[
    [
        'upper_biosample',
        'upper_assembly_accession',
        'upper_log2_mic',
        'upper_mic_mg_L',
        'neighbour_rank',
        'comparison_biosample',
        'comparison_assembly_accession',
        'comparison_log2_mic',
        'comparison_mic_mg_L',
        'delta_log2_mic_upper_minus_comparison',
        'K_relatedness',
        'K_derived_distance',
    ]
]

expected_nearest_rows = EXPECTED_HIGH_MIC * NEIGHBOURS_PER_UPPER

if len(nearest10) != expected_nearest_rows:
    raise ValueError(
        f'Expected {expected_nearest_rows} nearest-neighbour rows; '
        f'found {len(nearest10)}.'
    )

counts_per_upper = nearest10.groupby('upper_biosample').size()

if not (counts_per_upper == NEIGHBOURS_PER_UPPER).all():
    raise ValueError(
        'At least one upper-MIC pathogen does not have exactly 10 neighbours.'
    )

NEAREST10_PATH = (
    TABLE_DIRECTORY
    / '02_high_MIC_16_nearest_10_comparators.csv'
)

nearest10.to_csv(
    NEAREST10_PATH,
    index=False,
)

print('Nearest 10 chromosomal neighbours for each upper-MIC pathogen:')
display(nearest10)

print(f'Saved: {NEAREST10_PATH}')
print(
    'Transition: Cell 02.6 will summarize neighbour distance, MIC separation, '
    'and repeated use of the same comparison pathogens.'
)


In [ ]:
#@title Cell 02.6 - Summarize neighbour quality and MIC separation
# This cell summarizes the nearest-neighbour results without selecting final
# matched controls. It reports chromosomal distances, MIC differences and
# whether close lower-MIC comparators exist for each upper-MIC pathogen.

per_upper_rows = []

for upper_biosample, group in nearest10.groupby('upper_biosample'):
    group = group.sort_values('neighbour_rank')

    delta = group[
        'delta_log2_mic_upper_minus_comparison'
    ].to_numpy(dtype=float)

    distances = group[
        'K_derived_distance'
    ].to_numpy(dtype=float)

    per_upper_rows.append({
        'upper_biosample': upper_biosample,
        'upper_log2_mic': float(group['upper_log2_mic'].iloc[0]),
        'upper_mic_mg_L': float(group['upper_mic_mg_L'].iloc[0]),
        'nearest_distance': float(distances[0]),
        'median_distance_nearest10': float(np.median(distances)),
        'maximum_distance_nearest10': float(distances.max()),
        'nearest_neighbour_biosample': group['comparison_biosample'].iloc[0],
        'nearest_neighbour_mic_mg_L': float(group['comparison_mic_mg_L'].iloc[0]),
        'nearest_neighbour_delta_log2_mic': float(delta[0]),
        'n_nearest10_lower_MIC': int(np.sum(delta > 0.0)),
        'n_nearest10_at_least_1_log2_lower': int(np.sum(delta >= 1.0)),
        'n_nearest10_at_least_2_log2_lower': int(np.sum(delta >= 2.0)),
        'largest_delta_log2_mic_nearest10': float(delta.max()),
    })

per_upper_summary = (
    pd.DataFrame(per_upper_rows)
    .sort_values(
        ['upper_mic_mg_L', 'upper_biosample'],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

reuse_summary = (
    nearest10
    .groupby(
        [
            'comparison_biosample',
            'comparison_assembly_accession',
            'comparison_log2_mic',
            'comparison_mic_mg_L',
        ],
        dropna=False,
    )
    .agg(
        times_selected=('upper_biosample', 'size'),
        distinct_upper_pathogens=('upper_biosample', 'nunique'),
        minimum_distance=('K_derived_distance', 'min'),
        median_distance=('K_derived_distance', 'median'),
    )
    .reset_index()
    .sort_values(
        ['distinct_upper_pathogens', 'times_selected', 'minimum_distance'],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

overall_summary = pd.DataFrame([
    {
        'metric': 'Upper-MIC pathogens assessed',
        'value': EXPECTED_HIGH_MIC,
    },
    {
        'metric': 'Nearest neighbours examined per upper-MIC pathogen',
        'value': NEIGHBOURS_PER_UPPER,
    },
    {
        'metric': 'Upper-MIC pathogens with >=1 lower-MIC neighbour in nearest 10',
        'value': int((per_upper_summary['n_nearest10_lower_MIC'] >= 1).sum()),
    },
    {
        'metric': 'Upper-MIC pathogens with >=1 neighbour >=1 log2 lower',
        'value': int((per_upper_summary['n_nearest10_at_least_1_log2_lower'] >= 1).sum()),
    },
    {
        'metric': 'Upper-MIC pathogens with >=1 neighbour >=2 log2 lower',
        'value': int((per_upper_summary['n_nearest10_at_least_2_log2_lower'] >= 1).sum()),
    },
    {
        'metric': 'Unique comparison pathogens in nearest-10 sets',
        'value': int(nearest10['comparison_biosample'].nunique()),
    },
    {
        'metric': 'Maximum number of upper-MIC pathogens sharing one comparator',
        'value': int(reuse_summary['distinct_upper_pathogens'].max()),
    },
])

PER_UPPER_SUMMARY_PATH = (
    TABLE_DIRECTORY
    / '02_nearest_neighbour_quality_by_upper_MIC_pathogen.csv'
)
REUSE_SUMMARY_PATH = (
    TABLE_DIRECTORY
    / '02_repeated_nearest_neighbours.csv'
)
OVERALL_SUMMARY_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '02_nearest_neighbour_assessment_summary.csv'
)

per_upper_summary.to_csv(
    PER_UPPER_SUMMARY_PATH,
    index=False,
)
reuse_summary.to_csv(
    REUSE_SUMMARY_PATH,
    index=False,
)
overall_summary.to_csv(
    OVERALL_SUMMARY_PATH,
    index=False,
)

print('Neighbour quality by upper-MIC pathogen:')
display(per_upper_summary)

print('\nOverall nearest-neighbour assessment:')
display(overall_summary)

print('\nMost frequently reused comparison pathogens:')
display(reuse_summary.head(20))

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    per_upper_summary['nearest_distance'],
    per_upper_summary['nearest_neighbour_delta_log2_mic'],
)

for _, row in per_upper_summary.iterrows():
    ax.annotate(
        row['upper_biosample'],
        (
            row['nearest_distance'],
            row['nearest_neighbour_delta_log2_mic'],
        ),
        fontsize=7,
        xytext=(3, 3),
        textcoords='offset points',
    )

ax.axhline(
    0.0,
    linewidth=1,
    linestyle='--',
)

ax.set_xlabel('K-derived distance to nearest comparison pathogen')
ax.set_ylabel('Upper-MIC minus neighbour log2(MIC)')
ax.set_title(
    'Nearest chromosomal neighbour and MIC difference '
    'for the 16 upper-MIC pathogens'
)

fig.tight_layout()

NEAREST_NEIGHBOUR_FIGURE_PATH = (
    FIGURE_DIRECTORY
    / '02_nearest_comparator_distance_vs_MIC_difference.png'
)

fig.savefig(
    NEAREST_NEIGHBOUR_FIGURE_PATH,
    dpi=300,
    bbox_inches='tight',
)

plt.show()

print(f'Saved: {PER_UPPER_SUMMARY_PATH}')
print(f'Saved: {REUSE_SUMMARY_PATH}')
print(f'Saved: {OVERALL_SUMMARY_PATH}')
print(f'Saved: {NEAREST_NEIGHBOUR_FIGURE_PATH}')
print(
    '\nTransition: Cell 02.7 will save final QC, provenance and one complete '
    'Notebook 02 output ZIP. No final matched controls will be selected yet.'
)


In [ ]:
#@title Cell 02.7 - Save final QC, provenance and Notebook 02 output ZIP
# This cell records Notebook 02 QC and provenance and saves one complete ZIP.
# The notebook deliberately stops before selecting final matched controls.

QC_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '02_qc_summary.csv'
)
MANIFEST_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '02_manifest.json'
)
FINAL_ZIP_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '02_TEM1_High_MIC_Matched_Controls_outputs.zip'
)

final_qc = pd.DataFrame([
    {'metric': 'blaTEM-1-only pathogens', 'value': len(tem176)},
    {'metric': 'Upper-MIC pathogens', 'value': len(high16)},
    {'metric': 'Comparison pathogens', 'value': len(comparison160)},
    {'metric': 'Upper/comparison distance pairs', 'value': len(all_distances)},
    {'metric': 'Nearest neighbours retained per upper-MIC pathogen', 'value': NEIGHBOURS_PER_UPPER},
    {'metric': 'Nearest-neighbour table rows', 'value': len(nearest10)},
    {'metric': 'Upper-MIC pathogens represented', 'value': nearest10['upper_biosample'].nunique()},
    {'metric': 'K symmetric', 'value': bool(np.allclose(K_tem, K_tem.T))},
    {'metric': 'K finite', 'value': bool(np.isfinite(K_tem).all())},
    {'metric': 'Final matched controls selected', 'value': False},
])

final_qc.to_csv(
    QC_PATH,
    index=False,
)

manifest = {
    'notebook': '02_TEM1_High_MIC_Matched_Controls.ipynb',
    'project': 'Ceftazidime_Chromosomal_Evolution',
    'purpose': (
        'Assess chromosomally nearest comparator availability for the '
        '16 upper-MIC blaTEM-1-only pathogens.'
    ),
    'inputs': {
        'blaTEM_1_only_176': str(TEM176_PATH),
        'upper_MIC_16': str(HIGH_MIC_PATH),
        'comparison_160': str(COMPARISON_PATH),
        'K_matrix': str(RELATEDNESS_MATRIX_PATH),
        'K_pathogen_index': str(PATHOGEN_INDEX_PATH),
    },
    'method': {
        'distance_definition': 'sqrt(K_ii + K_jj - 2*K_ij)',
        'comparators_ranked_per_upper_MIC_pathogen': EXPECTED_COMPARISON,
        'nearest_neighbours_retained': NEIGHBOURS_PER_UPPER,
        'MIC_threshold_for_selection': None,
        'final_matched_controls_selected': False,
    },
    'decision_rule': (
        'Use the observed chromosomal distances and MIC differences from '
        'Notebook 02 to define the matching rule for the subsequent direct '
        'genome-comparison analysis.'
    ),
}

with open(
    MANIFEST_PATH,
    'w',
    encoding='utf-8',
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
    )

output_files = [
    ALL_DISTANCE_PATH,
    NEAREST10_PATH,
    PER_UPPER_SUMMARY_PATH,
    REUSE_SUMMARY_PATH,
    OVERALL_SUMMARY_PATH,
    NEAREST_NEIGHBOUR_FIGURE_PATH,
    QC_PATH,
    MANIFEST_PATH,
]

for path in output_files:
    if not path.exists():
        raise FileNotFoundError(
            f'Expected Notebook 02 output was not created: {path}'
        )

with zipfile.ZipFile(
    FINAL_ZIP_PATH,
    'w',
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in output_files:
        archive.write(
            path,
            arcname=str(path.relative_to(PROJECT_ROOT)),
        )

print('QC summary:')
display(final_qc)

print(f'\nSaved: {QC_PATH}')
print(f'Saved: {MANIFEST_PATH}')
print(f'Saved: {FINAL_ZIP_PATH}')

print(
    '\nNotebook 02 complete.\n'
    'No final matched controls were selected. '
    'Review Cell 02.6 before defining the comparator rule for direct genome comparison.'
)
